# Swarm pipeline — Colab setup & playground

This notebook brings up the stigmergic swarm on Colab and gives you a set of preset experiments to run.

**Storage split**
- **Google Drive (persistent):** run outputs, knowledge base, retrieval cache. Survives between sessions.
- **Ephemeral `/content/` (per-session):** cloned repo, HuggingFace cache, pip installs. Re-fetched every session.

**How updates work**
- The notebook you're reading right now is a **copy** in your Colab session. It does **not** auto-update when the GitHub repo changes.
- To get a newer notebook: `File → Open notebook → GitHub` → enter `sfuqua6/Stigmeric-Coordination` → pick `colab_setup.ipynb`. That opens a fresh copy with the latest cells.
- The **code** in `/content/swarm_repo/` is a `git clone` and *does* update — re-run cell 3 to `git pull`. So if you've only changed Python files (agents, core, configs), you don't need to reopen the notebook.
- If you change the notebook itself and want the changes to land in GitHub, copy them back manually (Colab doesn't push for you).

**Suggested flow on a fresh session**
1. Cells 1 → 5 in order (mount, paths, clone, install).
2. Cell 7 smoke test (`MOCK_LLM=1`) — proves plumbing in ~30 s.
3. Cell 9 real run, or any of the playground cells below.

Re-running this notebook top-to-bottom is the canonical "start a new session" workflow.

## 1. Mount Google Drive

Required: this is where outputs and the knowledge base will be persisted.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configure paths

Outputs / KB / retrieval cache → Drive. HuggingFace cache → ephemeral `/content/`.

In [ ]:
import os

# Persistent (Drive)
DRIVE_BASE = '/content/drive/MyDrive/swarm'
os.environ['SWARM_OUTPUTS_BASE_DIR']    = f'{DRIVE_BASE}/runs'
os.environ['SWARM_KB_DIR']              = f'{DRIVE_BASE}/knowledge_base'
os.environ['SWARM_RETRIEVAL_CACHE_DIR'] = f'{DRIVE_BASE}/retrieval_cache'

for d in ('runs', 'knowledge_base', 'retrieval_cache'):
    os.makedirs(f'{DRIVE_BASE}/{d}', exist_ok=True)

# Ephemeral (per-session, /content/)
os.environ['HF_HOME'] = '/content/hf_cache'
os.makedirs('/content/hf_cache', exist_ok=True)

# hf-xet has memory issues on some platforms; the standard downloader is fine.
os.environ['HF_HUB_DISABLE_XET'] = '1'

# Force the Colab tier-aware code paths. This makes core/config.py set
# _TIER = 't4' / 'l4' / 'a100_40' / 'a100_80' based on torch.cuda info,
# which in turn picks Qwen-7B-Instruct, vLLM as the backend, and
# concurrency 32. Without this the laptop defaults (DeepSeek + GGUF +
# concurrency 1) will be used and you will OOM.
os.environ['COLAB'] = '1'

print('Persistent (Drive):')
for k in ('SWARM_OUTPUTS_BASE_DIR', 'SWARM_KB_DIR', 'SWARM_RETRIEVAL_CACHE_DIR'):
    print(f'  {k} = {os.environ[k]}')
print(f"HF_HOME = {os.environ['HF_HOME']}")
print(f"COLAB   = {os.environ['COLAB']}")

## 3. Clone (or pull) the repository

`REPO_URL` is preset to the public Stigmeric-Coordination repo. If you've forked it, replace the URL with your fork (private repos: use a fine-grained PAT — `https://YOUR_PAT@github.com/USERNAME/REPO.git`).

Re-running this cell on a later session does `git pull` instead of a fresh clone — so if you've pushed new commits since you started this session, run this cell again to pick them up.

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/sfuqua6/Stigmeric-Coordination.git'
REPO_DIR = '/content/swarm_repo'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    print(f'{REPO_DIR} exists; running git pull')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
print('cwd =', os.getcwd())
print('contents:', sorted(os.listdir('.'))[:20])

## 4. Install dependencies

`vllm` is the heavy one (~5 min, brings its own CUDA-matched torch). The rest are retrieval, embedding, and the renderer-audit grounding lookups. All listed in `requirements-colab.txt`.

In [ ]:
!pip install -q -r requirements-colab.txt

# Quick CUDA sanity check
!python -c "import vllm; print(f'vllm {vllm.__version__} OK')"
!nvidia-smi --query-gpu=name,memory.free,memory.total --format=csv

## 5. Smoke test (MockLLM, no model load)

Verifies the pipeline + env vars before committing to a full real-model run. ~30 s. MockLLM emits SHA1-seeded phrases, so the *content* is gibberish — what you're checking is that the pipeline plumbing works end to end and writes a run directory to Drive.

In [ ]:
!MOCK_LLM=1 python run_swarm.py debate "Test thesis" --corpus=placeholder

## 6. Real run

Loads Qwen2.5-7B-Instruct via vLLM. On Colab T4 (16 GB VRAM) the loader walks a cascade — `core/llm.py:_build_cascade` — trying the configured settings first, then 4-bit AWQ, then a smaller 3B model, then HF/bnb. Watch the `[llm-cascade]` log lines: the first one to print `SUCCESS at attempt N/M` is what you actually loaded.

Expect ~90–120 min for a 3-round debate on T4, ~60–90 min on L4. Run directory lands at `$SWARM_OUTPUTS_BASE_DIR/outputs/debate_<timestamp>/`.

In [ ]:
!python run_swarm.py debate "Does free will exist?" 

## 7. Inspect outputs

Every run drops `answer.txt`, `signals.json`, `summary.json`, `round_log.json`, `citations.json`, `lineage.dot`, `run_meta.json` into a timestamped subdir. This cell prints the latest run's summary + answer.

In [ ]:
import json
from pathlib import Path

outputs_root = Path(os.environ['SWARM_OUTPUTS_BASE_DIR']) / 'outputs'
runs = sorted(outputs_root.glob('*'), key=lambda p: p.stat().st_mtime) if outputs_root.exists() else []

if runs:
    latest = runs[-1]
    print('latest run:', latest)
    print()
    summary_path = latest / 'summary.json'
    if summary_path.exists():
        print('=== summary.json ===')
        print(json.dumps(json.loads(summary_path.read_text()), indent=2))
        print()
    answer_path = latest / 'answer.txt'
    if answer_path.exists():
        print('=== answer.txt ===')
        print(answer_path.read_text())
else:
    print('no real runs yet — run cell 9, or check outputs_mock/ for cell 7 results')

---

# Playground

Below are preset experiments. Each cell is independent — once cells 1–5 have run you can hop around freely. Edit the prompts, flags, and env vars to probe different behavior. **Copy a cell (Ctrl+M, A) to fork an experiment without losing the preset.**

## P1. Try different task types

`run_swarm.py` supports five task types. Each gates a different set of roles on/off (see `ROLES_FOR_TASK` in `run_swarm.py`):

- `debate` — full pipeline incl. Hater and Validator
- `analysis` — full pipeline incl. Hater and Validator
- `problem_solving` — Validator suppressed (no shared external fact to verify)
- `creative` — Validator + Hater suppressed (a haiku has no consensus to challenge)
- `coding` — swaps in `agents/coding_roles.py` (RequirementsScout, StaticCritic, …)

Pick one and run. Use `MOCK_LLM=1` if you just want to see the role-activation log without burning GPU time.

In [ ]:
# Mock-mode tour of every task type (fast — ~30 s each).
# Drop the MOCK_LLM=1 to do a real run on whichever you actually care about.

!MOCK_LLM=1 python run_swarm.py problem_solving "How can cities reduce traffic?" --corpus=placeholder
# !MOCK_LLM=1 python run_swarm.py creative         "Write a haiku about emergence" --corpus=placeholder
# !MOCK_LLM=1 python run_swarm.py analysis         "What causes innovation?" --corpus=placeholder
# !MOCK_LLM=1 python run_swarm.py coding           "Implement a binary search" --corpus=placeholder

## P2. Toggle pipeline flags

| Flag | Effect |
|---|---|
| `--mode=baseline` | Disable signal store / partitioning / provenance boost. A/B comparison condition for the stigmergic hypothesis. |
| `--corpus=placeholder` | Skip web retrieval, use engineered corpus. Diversity numbers from this mode are not empirical evidence. |
| `--ignore-kb` | Don't consult the cross-run knowledge base. |
| `--reset-kb` | Quarantine existing KB entries before this run. |
| `--show-partition-overlap` | Surface Jaccard input-overlap diagnostics in the round log. |
| `--cloud-validator=anthropic` / `=gemini` | Use a cloud LLM for validation (requires `ANTHROPIC_API_KEY` / `GEMINI_API_KEY` env vars). |

Combine freely. The two cells below are paired stigmergic-vs-baseline runs over the same prompt — diff them with the comparison cell further down.

In [ ]:
# Stigmergic (default)
!MOCK_LLM=1 python run_swarm.py debate "Should AI development be paused?" \
    --corpus=placeholder --show-partition-overlap

In [ ]:
# Baseline (no signal store, no partitioning)
!MOCK_LLM=1 python run_swarm.py debate "Should AI development be paused?" \
    --mode=baseline --corpus=placeholder

## P3. Tweak the model / hardware envelope via env vars

These all live in `core/config.py` and `core/llm.py`. Set them with `%env` or by prefixing the `!python` command. **Restart and re-run cells 1–5 before changing model env vars** — `core.config` is a module-level `from config import *`, so values are baked in at import time.

| Env var | Default | What it does |
|---|---|---|
| `SWARM_MODEL` | tier-dependent (Qwen-7B on T4, 14B on L4, 32B on A100) | Override the model name |
| `SWARM_BACKEND` | auto (`vllm` on Colab, `gguf` if installed else `hf` on laptop) | Force a backend |
| `SWARM_AWQ_MODEL` | auto (`<base>-AWQ`) | Override the AWQ-quantized variant used in cascade stage 2 |
| `SWARM_GPU_MEM` | 13 GiB on T4 / 20 GiB on L4 / 36 GiB on A100-40 / 76 GiB on A100-80 | HF backend GPU budget; vLLM ignores this |
| `SWARM_CPU_MEM` | 30 GiB | Spillover budget |
| `SWARM_PROMPT_MAX_LEN` | 1024 | Tokenizer truncation cap |
| `VLLM_DTYPE` | float16 (T4/L4) / bfloat16 (A100) | vLLM weight dtype |
| `SWARM_QUIET_LIBS` | 1 | Set to `0` to re-enable transformers/HF chatter |
| `MOCK_LLM` | 0 | Skip model load entirely |

Tier detection (`_TIER`) is automatic from `torch.cuda.get_device_name(0)`. Force the Colab path on non-Colab hosts with `COLAB=1` (we already set this in cell 2).

In [ ]:
# Example: force a smaller model for a quick real run on T4.
# This skips the cascade-and-cleanup loop because the 3B fp16 model fits immediately.

!SWARM_MODEL='Qwen/Qwen2.5-3B-Instruct' python run_swarm.py debate \
    "Is consciousness computable?" 

In [ ]:
# Example: dial down the prompt window for a tighter KV cache.
# Useful if you're hitting OOM during generation rather than during load.

!SWARM_PROMPT_MAX_LEN=512 python run_swarm.py analysis \
    "What drives technological adoption?" 

## P4. Diagnose

`diagnose.py` runs a self-check on the signal store + pipeline plumbing. Useful when a real run produces empty outputs or NaN strengths — the diagnostic isolates the layer.

In [ ]:
!python diagnose.py

## P5. Compare two runs

`tools/compare_runs.py` diffs two `summary.json` files side-by-side. Most common use: compare a stigmergic run against its baseline twin (cells P2.A vs P2.B above).

In [ ]:
import os
from pathlib import Path

outputs_root = Path(os.environ['SWARM_OUTPUTS_BASE_DIR']) / 'outputs'
runs = sorted(outputs_root.glob('*'), key=lambda p: p.stat().st_mtime) if outputs_root.exists() else []

if len(runs) >= 2:
    a, b = runs[-2], runs[-1]
    print(f'comparing {a.name} vs {b.name}')
    !python tools/compare_runs.py "{a}" "{b}"
else:
    print(f'need at least 2 runs in {outputs_root}; have {len(runs)}')

## P6. Re-synthesize from a saved store

`synthesize.py` re-renders the final answer from a previously saved signal store. Useful when you want to swap the synthesizer model without re-running the entire pipeline.

In [ ]:
# Re-synthesize the most recent run (edit RUN_PATH to target a different one)
import os
from pathlib import Path

outputs_root = Path(os.environ['SWARM_OUTPUTS_BASE_DIR']) / 'outputs'
runs = sorted(outputs_root.glob('*'), key=lambda p: p.stat().st_mtime) if outputs_root.exists() else []
if runs:
    RUN_PATH = runs[-1]
    print(f'synthesizing from {RUN_PATH}')
    !python synthesize.py "{RUN_PATH}"
else:
    print('no runs yet')

## P7. Visualize the signal DAG

Every real run drops `lineage.dot` — a Graphviz file of the signal ancestry. Render it inline.

In [ ]:
# Render the most recent run's lineage DAG
import os, subprocess
from pathlib import Path
from IPython.display import Image, display

outputs_root = Path(os.environ['SWARM_OUTPUTS_BASE_DIR']) / 'outputs'
runs = sorted(outputs_root.glob('*'), key=lambda p: p.stat().st_mtime) if outputs_root.exists() else []
if runs and (runs[-1] / 'lineage.dot').exists():
    dot_path = runs[-1] / 'lineage.dot'
    png_path = runs[-1] / 'lineage.png'
    # graphviz is usually preinstalled on Colab; if not: !apt-get install -y graphviz
    subprocess.run(['dot', '-Tpng', str(dot_path), '-o', str(png_path)], check=True)
    display(Image(str(png_path)))
else:
    print('no runs with lineage.dot found')

## P8. Phase-isolated orchestrator (crash-resumable)

Spawns short-lived subprocesses (one per phase per round + synth). Each loads exactly one model, exits, and the next picks up the SignalStore checkpoint. Slower (subprocess startup overhead) but **crash-resumable** — if a phase fails, re-run the same command and it skips completed phases. Re-running across Colab session disconnects works too as long as the run dir is on Drive (which it is, via `SWARM_OUTPUTS_BASE_DIR`).

Pin a `--run-id` so re-runs land in the same directory.

In [ ]:
!python tools/run_isolated.py debate "Does free will exist?" --run-id=free_will_isolated

---

## (Optional, currently unused on Colab) GGUF model downloads

These ~33 GB GGUFs are only consumed by the `--heterogeneous` per-role routing path. On Colab `run_swarm.py` explicitly disables `--heterogeneous` (see `run_swarm.py:1171`) because vLLM single-model batching beats six quantized specialists each loaded once on T4/L4.

**Skip this section unless you've manually re-enabled heterogeneous routing.** Keeping the cell for symmetry with the laptop workflow.

In [ ]:
# OPTIONAL — only run if you've patched run_swarm.py to allow --heterogeneous on Colab.
# Downloads ~33 GB to /content/models/. Files disappear at session end.

from huggingface_hub import hf_hub_download
from pathlib import Path

MODELS_DIR = Path('/content/models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

MODELS = {
    'Qwen2.5-7B-Instruct-Q5_K_M.gguf':           ('bartowski/Qwen2.5-7B-Instruct-GGUF',           'Qwen2.5-7B-Instruct-Q5_K_M.gguf'),
    'Mistral-Nemo-Instruct-2407-Q4_K_M.gguf':    ('bartowski/Mistral-Nemo-Instruct-2407-GGUF',    'Mistral-Nemo-Instruct-2407-Q4_K_M.gguf'),
    'Phi-3.5-mini-instruct-Q4_K_M.gguf':         ('bartowski/Phi-3.5-mini-instruct-GGUF',         'Phi-3.5-mini-instruct-Q4_K_M.gguf'),
    'Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf':    ('bartowski/Meta-Llama-3.1-8B-Instruct-GGUF',    'Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf'),
    'DeepSeek-R1-Distill-Qwen-7B-Q4_K_M.gguf':   ('bartowski/DeepSeek-R1-Distill-Qwen-7B-GGUF',   'DeepSeek-R1-Distill-Qwen-7B-Q4_K_M.gguf'),
    'Qwen2.5-14B-Instruct-Q4_K_M.gguf':          ('bartowski/Qwen2.5-14B-Instruct-GGUF',          'Qwen2.5-14B-Instruct-Q4_K_M.gguf'),
}

for target, (repo, src) in MODELS.items():
    dst = MODELS_DIR / target
    if dst.exists() and dst.stat().st_size > 1_000_000_000:
        print(f'[skip] {target} already present ({dst.stat().st_size / 1e9:.2f} GB)')
        continue
    print(f'[get ] {repo}/{src} -> {dst.name}')
    path = hf_hub_download(repo_id=repo, filename=src, local_dir=str(MODELS_DIR))
    if Path(path).resolve() != dst.resolve():
        try: Path(path).rename(dst)
        except OSError: os.symlink(path, dst)

print('\nmodels directory:')
for f in sorted(MODELS_DIR.iterdir()):
    if f.is_file():
        print(f'  {f.name:50s}  {f.stat().st_size / 1e9:6.2f} GB')

---

## Troubleshooting

- **`/content/swarm_repo/run_swarm.py` doesn't exist.** Cell 3 failed silently. Check the printed output of cell 3 for `git clone` errors. The most common cause is a typo'd `REPO_URL` or hitting GitHub auth on a private repo without a PAT.
- **`[llm] all HF attempts failed; falling back to MockLLM`.** Look back through the `[llm-cascade]` log to see *which* attempt failed and *why* — OOM, network, version mismatch all look different. The pipeline still completes but the answer is SHA1-seeded gibberish. Don't trust outputs after this banner.
- **Colab session disconnects mid-run.** Use the phase-isolated orchestrator (cell P8) with a pinned `--run-id`. Checkpoints land on Drive under `$SWARM_OUTPUTS_BASE_DIR/outputs/<run_id>/`. Re-run the same command after reconnecting — it skips completed phases.
- **`llama-cpp-python` / `vllm` install fails.** CUDA wheel URLs change. If `pip install -r requirements-colab.txt` fails on `vllm`, try `!pip install --upgrade pip` first, then retry.
- **HuggingFace download stalls with a memory allocation error.** That's `hf-xet`. Cell 2 sets `HF_HUB_DISABLE_XET=1` to prevent this; if you still see it, `!pip uninstall -y hf-xet` and re-run.
- **VRAM OOM on the 14B synthesizer (L4 tier).** Set `SWARM_MODEL='Qwen/Qwen2.5-7B-Instruct'` before the run, or let the cascade in `core/llm.py` step down to AWQ / 3B automatically.
- **Drive quota for outputs.** A 3-round debate run is ~5–15 MB of JSON. Even 100 runs is under 2 GB.
- **Idle disconnect.** Free Colab disconnects after ~90 min of inactivity. Keep the tab focused, or in a browser DevTools console:
  ```javascript
  setInterval(() => document.querySelector('colab-toolbar-button#connect')?.click(), 60000);
  ```
- **I updated the notebook on GitHub — why isn't my Colab seeing it?** Notebooks aren't auto-synced. `File → Open notebook → GitHub` to re-open a fresh copy. The cloned *code* under `/content/swarm_repo/` does pull via cell 3.